# 🎨 Qwen-Image-2.1 高画质图像生成与编辑引擎 (Colab A100 版)
> **专为 Antigravity 自动化调度与交互式探索设计的无头 (Headless) 引擎**
- **模型架构**：7B 单流块因果 Diffusion Transformer (DiT) + Qwen3-VL 文本/多模态编码器 + 64通道原生透明 RGBA VAE
- **统一多任务**：原生支持高质量文生图 (T2I)、图像编辑 / 多参考图控制 (I2I) 以及透明图层生成
- **加速后端**：基于 DiffSynth-Studio 极速推理与显存自适应调度，完全驻留 A100 (40GB/80GB) 显存

## 1. 硬件检测与显存预算

In [ ]:
import os, sys, torch
assert torch.cuda.is_available(), '❌ 未检测到 GPU！请在 Colab 运行时分配 A100。'

gpu_name = torch.cuda.get_device_name(0)
free_mem, total_mem = torch.cuda.mem_get_info('cuda')
vram_gb = total_mem / (1024 ** 3)
vram_limit = max(vram_gb - 2.0, 14.0)

print(f'🚀 GPU 型号: {gpu_name}')
print(f'💾 显存总量: {vram_gb:.1f} GB (当前空闲: {free_mem/(1024**3):.1f} GB)')
print(f'🎯 引擎显存预算: {vram_limit:.1f} GB')

## 2. 依赖安装与 DiffSynth-Studio 准备

In [ ]:
if not os.path.exists('/content/DiffSynth-Studio'):
    print('📦 正在克隆 DiffSynth-Studio 源码...')
    !git clone --depth 1 https://github.com/modelscope/DiffSynth-Studio.git /content/DiffSynth-Studio
    !pip install -q -e /content/DiffSynth-Studio
    !pip install -q -U bitsandbytes hf_transfer transformers accelerate safetensors sentencepiece huggingface_hub torchvision

if '/content/DiffSynth-Studio' not in sys.path:
    sys.path.insert(0, '/content/DiffSynth-Studio')

os.environ['DIFFSYNTH_DOWNLOAD_SOURCE'] = 'HuggingFace'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('✅ 运行环境就绪！')

## 3. Qwen-Image-2.1 Pipeline 模型热装载

In [ ]:
from diffsynth.pipelines.qwen_image_21 import QwenImage21Pipeline, ModelConfig

vram_config = {
    'offload_dtype': torch.bfloat16,
    'offload_device': 'cpu',
    'onload_dtype': torch.bfloat16,
    'onload_device': 'cpu',
    'preparing_dtype': torch.bfloat16,
    'preparing_device': 'cuda',
    'computation_dtype': torch.bfloat16,
    'computation_device': 'cuda',
}

model_configs = [
    ModelConfig(model_id='Qwen/Qwen-Image-2.1', origin_file_pattern='transformer/diffusion_pytorch_model*.safetensors', **vram_config),
    ModelConfig(model_id='Qwen/Qwen-Image-2.1', origin_file_pattern='text_encoder/model*.safetensors', **vram_config),
    ModelConfig(model_id='Qwen/Qwen-Image-2.1', origin_file_pattern='vae/diffusion_pytorch_model*.safetensors', **vram_config),
]

processor_config = ModelConfig(model_id='Qwen/Qwen-Image-2.1', origin_file_pattern='processor/')

print('⏳ 开始装载 Qwen-Image-2.1 Pipeline...')
pipe = QwenImage21Pipeline.from_pretrained(
    torch_dtype=torch.bfloat16,
    device='cuda',
    model_configs=model_configs,
    processor_config=processor_config,
    vram_limit=vram_limit,
)
print('🎉 Qwen-Image-2.1 Pipeline 已成功装载入 A100 显存！')

## 4. 标准化生产级生图 API

In [ ]:
import datetime, json
from PIL import Image

def align_to_32(val, min_val=256, max_val=2048):
    val = max(min_val, min(max_val, int(val)))
    return (val + 31) // 32 * 32

def load_as_pil(img_or_path):
    if img_or_path is None:
        return None
    if isinstance(img_or_path, str):
        return Image.open(img_or_path).convert('RGB')
    return img_or_path.convert('RGB')

def generate_qwen_image(
    prompt: str,
    negative_prompt: str = None,
    ref_image = None,
    width: int = 1024,
    height: int = 1024,
    num_inference_steps: int = 40,
    cfg_scale: float = 4.5,
    seed: int = 42,
    output_dir: str = '/content/outputs'
):
    if 'pipe' not in globals() or globals()['pipe'] is None:
        raise RuntimeError('QwenImage21Pipeline 未就绪！')

    width = align_to_32(width)
    height = align_to_32(height)

    os.makedirs(f'{output_dir}/generated_images', exist_ok=True)
    os.makedirs(f'{output_dir}/generation_logs', exist_ok=True)
    os.makedirs(f'{output_dir}/previews', exist_ok=True)

    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    png_path = f'{output_dir}/generated_images/qwen_{ts}.png'
    preview_path = f'{output_dir}/previews/preview_qwen_{ts}.jpg'
    log_path = f'{output_dir}/generation_logs/qwen_{ts}.json'

    pil_ref = load_as_pil(ref_image)
    if pil_ref is not None:
        pil_ref = pil_ref.resize((width, height), Image.LANCZOS)

    kwargs = {
        'prompt': prompt,
        'width': width,
        'height': height,
        'num_inference_steps': num_inference_steps,
        'seed': seed,
    }
    if negative_prompt:
        kwargs['negative_prompt'] = negative_prompt
    if cfg_scale is not None:
        kwargs['cfg_scale'] = cfg_scale
    if pil_ref is not None:
        kwargs['input_image'] = pil_ref

    mode = 'I2I' if pil_ref is not None else 'T2I'
    print(f'[Render Start] Mode: {mode} | Res: {width}x{height} | Steps: {num_inference_steps} | Seed: {seed}', flush=True)

    import time
    t0 = time.time()
    img = pipe(**kwargs)
    infer_sec = round(time.time() - t0, 2)

    img.save(png_path)
    preview = img.convert('RGB')
    preview.thumbnail((512, 512))
    preview.save(preview_path, format='JPEG', quality=85)

    metadata = {
        'timestamp': ts,
        'prompt': prompt,
        'negative_prompt': negative_prompt,
        'mode': mode,
        'width': width,
        'height': height,
        'num_inference_steps': num_inference_steps,
        'cfg_scale': cfg_scale,
        'seed': seed,
        'inference_time_sec': infer_sec,
        'image_path': png_path,
        'preview_path': preview_path,
        'format': img.mode
    }
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f'[Render Complete] Saved to {png_path} ({infer_sec}s)')
    return metadata

print('✅ generate_qwen_image 函数已定义完毕。')

## 5. 单图测试生成

In [ ]:
meta = generate_qwen_image(
    prompt='电影级质感，1930年代维多利亚式暗色书房，壁炉中微弱的火光映照在红木书桌上，上面放着一封未拆封的带血蜡封信件，窗外大雨倾盆，极高细节，8k分辨率。',
    width=1024,
    height=1024,
    num_inference_steps=40,
    seed=42
)
from IPython.display import display
display(Image.open(meta['image_path']))